# 02_API_test_noteboook

- this is for the test env. for the leacture : Building with the Claude API served by Anthropic Academy

## 0. Building the environment

- 05_making_a_request

In [2]:
!python --version

Python 3.14.5


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [18]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [20]:
message = client.messages.create(
    model=model,
    max_tokens=100,
    messages=[
        {
            "role" : "user",
            "content": "say hello there!"
        }
    ]
)

In [21]:
message.content[0].text

'Hello there! 👋\n\nHow can I help you today?'

In [16]:
message

Message(id='msg_011CddaFu1ur6woSBcXq4byA', container=None, content=[TextBlock(citations=None, text='Hello there! 👋', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=11, output_tokens=9, output_tokens_details=None, server_tool_use=None, service_tier='standard'))

## 1. Let's get started!

### Accessubg Claude with the API

### 06_Multi-Turn_conversations

In [4]:
from typing import Optional


def add_user_message(
    messages : list[Optional[dict[str, str]]],
    text: str
):
    user_message = {
        "role" : "user",
        "content" : text
    }
    messages.append(user_message)

def add_assistant_message(
    messages: list[Optional[dict[str, str]]],
    text: str
):
    assistant_message = {
        "role" : "assistant",
        "content" : text
    }
    messages.append(assistant_message)

def chat(
    messages: list[Optional[dict[str, str]]]
):
    message = client.messages.create(
        model=model,
        max_tokens=100,
        messages=messages
    )
    return message.content[0].text

In [25]:
messages = []

add_user_message(
    messages=messages,
    text="忘れないための勉強方法は? in one sentence."
)
llm_response = chat(messages)
llm_response

'# 記憶定着の勉強方法\n\n**定期的な復習（特に1日後、1週間後、1ヶ月後）と、学んだ内容を実際に使う・教えるなどのアウトプットを組み合わせることが最も効果的です。**'

In [26]:
add_assistant_message(
    messages=messages,
    text=llm_response
)
add_user_message(
    messages=messages,
    text="エビングワースの忘却曲線はどう？ in one sentence."
)
llm_response2 = chat(
    messages=messages
)
llm_response2

'# エビングワースの忘却曲線について\n\n**忘却曲線は時間とともに記憶が急速に低下することを示しており、この理論に基づいて復習のタイミングを計画する「間隔反復学習」が効果的な勉強法として広く活用されています。**'

### 08_System_prompts


In [34]:
def chat(
    messages: list[Optional[dict[str, str]]],
    system: Optional[str] = None
):
    params = {
        "model":model,
        "max_tokens":100,
        "messages":messages
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)

    return message.content[0].text

In [33]:
messages

[{'role': 'user', 'content': '忘れないための勉強方法は? in one sentence.'},
 {'role': 'assistant',
  'content': '# 記憶定着の勉強方法\n\n**定期的な復習（特に1日後、1週間後、1ヶ月後）と、学んだ内容を実際に使う・教えるなどのアウトプットを組み合わせることが最も効果的です。**'},
 {'role': 'user', 'content': 'エビングワースの忘却曲線はどう？ in one sentence.'}]

In [ ]:
answer1 = chat(messages)
answer1

'# エビングワースの忘却曲線について\n\n**忘却曲線は、復習のタイミング（1日後、1週間後、1ヶ月後など）を科学的に示しており、その時点での復習が記憶定着を劇的に高めるため、効率的な学習計画の基礎として非常に有用です'

In [40]:
system = """
あなたは英語教師です。英語の勉強の視点で具体的な手順を示す必要があります。
"""

answer2 = chat(messages=messages, system=system)

In [41]:
answer2

'# エビングワースの忘却曲線について\n\n**学習後1日目、1週間後、1ヶ月後に復習することで記憶の定着率が大きく向上するという科学的根拠があり、英語学習では特に単語や文法を繰り返し復習する際の最適なタイミングを示してくれます'

### 10_Tempature

In [42]:
def chat(
    messages: list[Optional[dict[str, str]]],
    system: Optional[str] = None,
    temperature: Optional[float] = 1.0
):
    params = {
        "model":model,
        "max_tokens":100,
        "messages":messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)

    return message.content[0].text

In [47]:
# Low temperature - more predictable
answer_low = chat(messages, temperature=0.0)
answer_low

'# エビングワースの忘却曲線\n\n**忘却曲線は、学習直後から急速に忘れ始め、復習のタイミングを最適化することで記憶を長期保持できることを示しており、効率的な学習スケジュール（1日後、3日後、1週間後など）の根拠となっています。'

In [ ]:
# High temperature - more creative
answer_high = chat(messages, temperature=1.0)
answer_high

'# エビングハウスの忘却曲線\n\n**人間は学習直後から急速に忘れ始めるが、定期的な復習によって忘却速度が遅くなるという理論で、これに基づいた間隔反復学習が記憶定着に非常に効果的です。**'

: 

### 12_Response streaming

In [ ]:
messages = []
add_user_message(
    messages=messages,
    text="Claude Certified Architect – Professional についてCCA_F,CCAR_Fなどとの違いを1sentenceで教えて"
)

In [12]:
stream = client.messages.create(
    model=model,
    max_tokens=100,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_011CdfqdLC5SZScLmhyTH8oi', container=None, content=[], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=41, output_tokens=1, output_tokens_details=None, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='#', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' Claude認定資格の違い\n\nClaude Certified Architect – Professional (CCA-P)', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(

In [19]:
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")

    final_message = stream.get_final_message()

# Claude Certified Architect – Professional の位置づけ

Claude Certified Architect – Professional は、**Anthropic社によるClaudeの認定資格体系における最高レベルの認定資格**で、CCA_F（Foundation）やCCAR_F（Associate Foundation）などの初級・中級資格よりも、Claudeの高度な機能・プロンプトエンジニアリング・エンタープライズ導入に関する実践的な知識と技能を要求する上級資格です。

In [20]:
final_message

ParsedMessage(id='msg_011Cdfqsuo3rGe9s7k21zDfG', container=None, content=[ParsedTextBlock(citations=None, text='# Claude Certified Architect – Professional の位置づけ\n\nClaude Certified Architect – Professional は、**Anthropic社によるClaudeの認定資格体系における最高レベルの認定資格**で、CCA_F（Foundation）やCCAR_F（Associate Foundation）などの初級・中級資格よりも、Claudeの高度な機能・プロンプトエンジニアリング・エンタープライズ導入に関する実践的な知識と技能を要求する上級資格です。', type='text', parsed_output=None)], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=41, output_tokens=142, output_tokens_details=None, server_tool_use=None, service_tier='standard'))

In [ ]:
final_message.content[0].text

'# Claude Certified Architect – Professional の位置づけ\n\nClaude Certified Architect – Professional は、**Anthropic社によるClaudeの認定資格体系における最高レベルの認定資格**で、CCA_F（Foundation）やCCAR_F（Associate Foundation）などの初級・中級資格よりも、Claudeの高度な機能・プロンプトエンジニアリング・エンタープライズ導入に関する実践的な知識と技能を要求する上級資格です。'

: 

### 13_Structured_data

In [22]:
messages = []
add_user_message(
    messages=messages,
    text="もっとも簡単な構成のmcp_server.pyのコードを教えて"
)

In [7]:
answer = client.messages.create(
    messages=messages,
    max_tokens=100,
    model=model
)
answer

Message(id='msg_011Cdg3EXSWQ8Uu78GW2kFn9', container=None, content=[TextBlock(citations=None, text='# 最もシンプルなMCP Serverのコード\n\n```python\nimport asyncio\nimport json\nfrom typing import Any\n\nclass MCPServer:\n    def __init__(self):\n        self.tools = {}\n    \n    def add_tool(self, name: str, func, description: str = ""):\n        """ツールを登録"""\n        self.tools[name] = {"func": func, "', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='max_tokens', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=31, output_tokens=100, output_tokens_details=None, server_tool_use=None, service_tier='standard'))

In [8]:
answer.content[0].text

'# 最もシンプルなMCP Serverのコード\n\n```python\nimport asyncio\nimport json\nfrom typing import Any\n\nclass MCPServer:\n    def __init__(self):\n        self.tools = {}\n    \n    def add_tool(self, name: str, func, description: str = ""):\n        """ツールを登録"""\n        self.tools[name] = {"func": func, "'

In [14]:
add_assistant_message(messages,text="```python")
answer_python = client.messages.create(
    messages=messages,
    max_tokens=1000,
    model=model,
    stop_sequences=["```"]
)
answer_python

Message(id='msg_011Cdg3XMhqYpHMnQpbXu2QY', container=None, content=[TextBlock(citations=None, text='\nimport json\nimport sys\nfrom typing import Any\n\ndef process_request(request: dict) -> dict:\n    """MCPリクエストを処理"""\n    method = request.get("method")\n    \n    if method == "initialize":\n        return {\n            "protocolVersion": "2024-11",\n            "capabilities": {},\n            "serverInfo": {\n                "name": "simple-mcp-server",\n                "version": "1.0.0"\n            }\n        }\n    \n    elif method == "resources/list":\n        return {\n            "resources": []\n        }\n    \n    elif method == "tools/list":\n        return {\n            "tools": [\n                {\n                    "name": "hello",\n                    "description": "あいさつ",\n                    "inputSchema": {\n                        "type": "object",\n                        "properties": {\n                            "name": {"type": "string"}\n           

In [15]:
answer_python.content[0].text.replace('\n', '')

'import jsonimport sysfrom typing import Anydef process_request(request: dict) -> dict:    """MCPリクエストを処理"""    method = request.get("method")        if method == "initialize":        return {            "protocolVersion": "2024-11",            "capabilities": {},            "serverInfo": {                "name": "simple-mcp-server",                "version": "1.0.0"            }        }        elif method == "resources/list":        return {            "resources": []        }        elif method == "tools/list":        return {            "tools": [                {                    "name": "hello",                    "description": "あいさつ",                    "inputSchema": {                        "type": "object",                        "properties": {                            "name": {"type": "string"}                        }                    }                }            ]        }        elif method == "tools/call":        tool_name = request.get("params", {}).get("name")

In [ ]:
import json
import sys
from typing import Any

def process_request(request: dict) -> dict:
    """MCPリクエストを処理"""
    method = request.get("method")
    if method == "initialize":
        return {
            "protocolVersion": "2024-11",
            "capabilities": {},
            "serverInfo": {
                "name":"simple-mcp-server",
                "version": "1.0.0"
            }
        }
    elif method == "resources/list":
        return {
            "resources": []
        }
    elif method == "tools/list":
        return {
            "tools": [
                {
                    "name": "hello",
                    "description": "あいさつ",
                    "inputSchema":
                        {
                            "type": "object",
                            "properties":
                                {
                                    "name": {
                                            "type":"string"
                                        }
                                }
                            }
                    }
            ]
        }
    elif method == "tools/call":
        tool_name = request.get("params", {}).get("name")
        if tool_name == "hello":
            name = request.get("params", {}).get("arguments", {}).get("name", "World")
            return {
                "content": [
                    {
                        "type": "text",
                        "text": f"Hello, {name}!"
                    }
                ]
            }
        return {"error": "Unknown method"}

def main():
    while True:
        try:
            line = sys.stdin.readline()
            if not line:
                break
            request = json.loads(line)
            response = process_request(request)
            print(json.dumps(response))
            sys.stdout.flush()
        except Exception as e:
            error_response = {"error": str(e)}
            print(json.dumps(error_response))
            sys.stdout.flush()

if __name__ == "__main__":
    main()

In [27]:
messages = []
add_user_message(
    messages=messages,
    text="スピーキングに必要なことを3点箇条書きで教えて"
)

add_assistant_message(messages, text="1.")
answer_list = client.messages.create(
    messages=messages,
    max_tokens=1000,
    model=model,
    stop_sequences=["\\n"]
)
answer_list


Message(id='msg_011Cdg6g7cNDgAZ5RHNUx6ca', container=None, content=[TextBlock(citations=None, text='語彙や文法の基礎知識\n    実際に使える単語や表現を身につけておくこと\n\n2.継続的な発話練習\n    実際に声に出す、独り言、会話練習など繰り返すこと\n\n3.聴く力（リスニング）\n    他者の発話を理解し、自然な流れで応答すること', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=32, output_tokens=107, output_tokens_details=None, server_tool_use=None, service_tier='standard'))

In [ ]:
answer_list.content[0].text

'語彙や文法の基礎知識\n    実際に使える単語や表現を身につけておくこと\n\n2.継続的な発話練習\n    実際に声に出す、独り言、会話練習など繰り返すこと\n\n3.聴く力（リスニング）\n    他者の発話を理解し、自然な流れで応答すること'

: 

1. 語彙や文法の基礎知識
    - 実際に使える単語や表現を身につけておくこと
2. 継続的な発話練習
    - 実際に声に出す、独り言、会話練習など繰り返すこと
3. 聴く力（リスニング）
    - 他者の発話を理解し、自然な流れで応答すること

## 31-

In [ ]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"